# PointSegNet Pipeline - Offline Precomputation

This notebook handles Phase 1 (Data Parsing) and Phase 2 (Offline KNN Precomputation) for a custom PointSegNet pipeline. 
It processes raw ASCII `.pcd` files into cached, preprocessed `.npz` files ready for a PyTorch DataLoader.

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import os
from scipy.spatial import cKDTree
import time

In [ ]:
# Placeholder dictionary for mapping original simulator IDs to super-classes
# Super-classes:
# 0 = Terrain (Roads, ground, flat surfaces)
# 1 = Static Obstacles (Buildings, poles, walls, trees)
# 2 = Dynamic Objects (Vehicles, pedestrians, cyclists)
# 
# Using the mapping from the LidarAnalysis dataset as an example:
# 0: Unlabelled (assumed terrain/ground for this test)
# 1: Car, 2: Truck, 3: Bicycle, 4: Pedestrian -> Dynamic (2)
# 5: Jersey Barrier, 6: Guardrail -> Static (1)

LABEL_MAPPING = {
    0: 0, # Unlabelled -> Terrain
    1: 2, # Car -> Dynamic
    2: 2, # Truck -> Dynamic
    3: 2, # Bicycle -> Dynamic
    4: 2, # Pedestrian -> Dynamic
    5: 1, # Jersey Barrier -> Static
    6: 1  # Guardrail -> Static
}

def remap_labels(raw_labels, mapping_dict):
    """
    Maps raw simulator labels to 3 super-classes using a mapping dictionary.
    """
    # Map each label, defaulting to 0 (Terrain) if not found in mapping
    remapped = np.array([mapping_dict.get(int(l), 0) for l in raw_labels], dtype=np.int32)
    return remapped

def parse_ascii_pcd(filepath):
    """
    Parses an ASCII .pcd file and extracts x, y, z, intensity, and label.
    Assuming standard PCD headers of 11 lines and fields: x y z intensity actor_id class_id material_id
    """
    print(f"Parsing {filepath}...")
    try:
        # We use pandas for fast CSV parsing, skipping the 11-line header
        df = pd.read_csv(filepath, sep=' ', skiprows=11, header=None, engine='c', 
                         names=['x', 'y', 'z', 'intensity', 'actor_id', 'class_id', 'material_id'])
        
        # Drop rows where coordinates are NaN
        df = df.dropna(subset=['x', 'y', 'z'])
        
        # Extract features
        coords = df[['x', 'y', 'z']].values.astype(np.float32)
        intensity = df['intensity'].values.astype(np.float32)
        raw_labels = df['class_id'].values
        
        return coords, intensity, raw_labels
    except Exception as e:
        print(f"Error parsing {filepath}: {e}")
        return None, None, None

In [ ]:
def extract_features_and_neighbors(coords, intensity, k=16):
    """
    Calculates spherical features and performs KNN search.
    Returns the concatenated feature array and the indices of the K nearest neighbors.
    """
    print("Calculating spherical coordinates (r, theta)...")
    x = coords[:, 0]
    y = coords[:, 1]
    z = coords[:, 2]
    
    # Calculate radial distance r and azimuth angle theta
    r = np.sqrt(x**2 + y**2 + z**2)
    theta = np.arctan2(y, x)
    
    # Concatenate to form [N, 6] features: x, y, z, intensity, r, theta
    features = np.column_stack((x, y, z, intensity, r, theta)).astype(np.float32)
    
    print(f"Building KDTree for {len(coords)} points...")
    start_time = time.time()
    tree = cKDTree(coords)
    
    print(f"Querying k={k} nearest neighbors...")
    # Query the tree. Returns distances and indices.
    # Note: The first neighbor is the point itself (distance 0).
    distances, neighbor_idx = tree.query(coords, k=k, workers=-1) 
    print(f"KNN computation took {time.time() - start_time:.2f} seconds.")
    
    return features, neighbor_idx.astype(np.int32)

In [ ]:
def save_precomputed_data(features, labels, neighbor_idx, output_path):
    """
    Saves the precomputed arrays to a compressed .npz file.
    """
    print(f"Saving to {output_path}...")
    np.savez_compressed(
        output_path,
        features=features,
        labels=labels,
        neighbor_idx=neighbor_idx
    )
    print("Save complete!")
    
    # Verify the saved file sizes
    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"Output file size: {file_size_mb:.2f} MB")

In [ ]:
def process_file(input_filepath, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    # Extract base filename
    filename = os.path.basename(input_filepath)
    name, _ = os.path.splitext(filename)
    output_path = os.path.join(output_dir, f"{name}_preprocessed.npz")
    
    # 1. Parse
    coords, intensity, raw_labels = parse_ascii_pcd(input_filepath)
    if coords is None:
        return
        
    # Remap labels
    remapped_labels = remap_labels(raw_labels, LABEL_MAPPING)
    
    # 2. KNN Precomputation
    features, neighbor_idx = extract_features_and_neighbors(coords, intensity, k=16)
    
    # 3. Save
    save_precomputed_data(features, remapped_labels, neighbor_idx, output_path)
    
    # 4. Verify Shapes
    print("\n--- Verification ---")
    print(f"Features shape: {features.shape} (Expected: [N, 6])")
    print(f"Labels shape: {remapped_labels.shape} (Expected: [N])")
    print(f"Neighbor Indices shape: {neighbor_idx.shape} (Expected: [N, 16])")
    
    # Let's load it back just to be absolutely sure
    loaded = np.load(output_path)
    print(f"Loaded keys: {list(loaded.keys())}")
    
# --- Execution on Sample Data ---
# Update this path to point to a valid PCD file from your dataset
DATASET_PATH = '../../lidar_dataset/Exported_PCD_Files_ryan_Final/frame_0001.pcd'
OUTPUT_DIR = './processed_output'

if os.path.exists(DATASET_PATH):
    process_file(DATASET_PATH, OUTPUT_DIR)
else:
    print(f"File not found: {DATASET_PATH}")
    print("Please update DATASET_PATH to a valid PCD file.")

## Step 5: PointSegNet PyTorch Architecture

Here we define the PyTorch model architecture for the PointSegNet custom point cloud segmentation network. It handles the forward pass for features and `neighbor_idx` arrays.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PointSegNet(nn.Module):
    def __init__(self, in_channels=6, num_classes=3):
        super(PointSegNet, self).__init__()
        
        # Feature Extraction (Shared MLP over neighbors)
        # We use Conv2d with 1x1 kernels as a shared MLP over [Batch, Channels, N, K]
        # Input channels is 2 * in_channels because we concatenate center features and relative features
        self.conv1 = nn.Conv2d(in_channels * 2, 64, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(64)
        
        self.conv2 = nn.Conv2d(64, 128, kernel_size=1)
        self.bn2 = nn.BatchNorm2d(128)
        
        # Classification Head (Shared MLP per point)
        # After pooling out the neighbors, we use Conv1d over [Batch, Channels, N]
        self.conv3 = nn.Conv1d(128, 64, kernel_size=1)
        self.bn3 = nn.BatchNorm1d(64)
        
        self.conv4 = nn.Conv1d(64, num_classes, kernel_size=1)
        
    def gather_neighbors(self, features, neighbor_idx):
        """
        Extracts neighbor features using advanced indexing.
        features shape: [B, N, C]
        neighbor_idx shape: [B, N, K]
        Returns: [B, N, K, C]
        """
        B, N, C = features.shape
        # Create batch indices of shape [B, 1, 1] which broadcasts automatically
        batch_indices = torch.arange(B, dtype=torch.long, device=features.device).view(B, 1, 1)
        gathered_features = features[batch_indices, neighbor_idx, :]
        return gathered_features

    def forward(self, features, neighbor_idx):
        # 1. Neighbor Gathering
        gathered_features = self.gather_neighbors(features, neighbor_idx) # [B, N, K, C]
        
        # 2. Relative Features
        # Expand center features to match the K dimension
        center_features = features.unsqueeze(2).expand(-1, -1, neighbor_idx.shape[-1], -1) # [B, N, K, C]
        relative_features = gathered_features - center_features
        
        # Concatenate original center features with relative features
        combined_features = torch.cat([center_features, relative_features], dim=-1) # [B, N, K, 2C]
        
        # 3. Feature Extraction
        # Conv2d requires [Batch, Channels, H, W], so we permute to [B, Channels, N, K]
        x = combined_features.permute(0, 3, 1, 2)
        
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x))) # [B, 128, N, K]
        
        # 4. Max Pooling
        # Apply torch.max across the neighbor dimension (k=16) -> dim=3
        x = torch.max(x, dim=3)[0] # [B, 128, N]
        
        # 5. Classification Head
        x = F.relu(self.bn3(self.conv3(x))) # [B, 64, N]
        logits = self.conv4(x) # [B, 3, N]
        
        # Output is typically [Batch, N, Classes] for point cloud segmentation loss
        return logits.permute(0, 2, 1)

# --- Test Block ---
# Verifying dimensions matching the preprocessed NPZ outputs
if __name__ == '__main__':
    B, N, K, C = 1, 1000, 16, 6
    dummy_features = torch.randn(B, N, C)
    dummy_neighbor_idx = torch.randint(0, N, (B, N, K))
    
    # Initialize Model
    model = PointSegNet(in_channels=C, num_classes=3)
    
    # Forward Pass
    output = model(dummy_features, dummy_neighbor_idx)
    
    print("--- PyTorch Model Architecture Test ---")
    print(f"Input features shape: {dummy_features.shape}")
    print(f"Input neighbor_idx shape: {dummy_neighbor_idx.shape}")
    print(f"Output shape: {output.shape} (Expected: [1, 1000, 3])")
